# Nes2Net: tái hiện paper (Kaggle · Colab · local)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trinhphamtrunghieu/nes2net-fake-voice-detection/blob/main/notebooks/nes2net.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/trinhphamtrunghieu/nes2net-fake-voice-detection/blob/main/notebooks/nes2net.ipynb)

**Paper:** *Nes2Net: A Lightweight Nested Architecture for Foundation Model Driven Speech Anti-spoofing*
(Liu, Truong, Das, Lee, Li, IEEE T-IFS 2025, [arXiv:2504.05657](https://arxiv.org/abs/2504.05657)).

Một notebook cho mọi nền tảng: notebook tự nhận Kaggle / Colab / local và chọn chỗ lưu (`/kaggle/working`,
Google Drive, `./work`). Code nằm trong package `nes2net/`; notebook chỉ chọn **preset** và gọi các hàm của nó.

| Preset | Chạy gì | Thời gian (T4) |
|---|---|---|
| `smoke` | tập con rất nhỏ của mọi phần, chỉ để kiểm tra pipeline | ~30 phút |
| `default` | **A** checkpoint chính thức trên tập con · **C** ablation · **C2** chọn layer SSL | ~3–4 giờ |
| `full_eval` | **A** trên toàn bộ LA19 / ITW / 2021 LA / 2021 DF · **C** · **C2** | ~6–8 giờ |
| `vi_zeroshot` | checkpoint chính thức, zero-shot trên tập thử **tiếng Việt** (FLEURS vi so với cùng câu do MMS-TTS đọc) + tập đối chứng tiếng Anh | ~30–45 phút |
| `diag_data` | kiểm tra dữ liệu train ASVspoof2019 LA: nhãn so với attack id, thống kê audio theo split/lớp, EER của checkpoint chính thức trên train/dev/test | ~15 phút |
| `paper_train` | **B** train đúng công thức paper (lr 2.5e-7, bs 12, 100 epoch, RawBoost 4), resume qua nhiều phiên | ~80–100 giờ |

**Chuẩn bị:** GPU (Kaggle: T4, Internet **On**; Colab: *Runtime → Change runtime type → GPU*, Pro nên chọn A100/L4).
Trên Colab, ô Setup xin quyền mount Google Drive để lưu kết quả và trạng thái resume.
Dữ liệu tải từ Hugging Face Hub (`Bisher/ASVspoof_2019_LA`, `SpeechAntiSpoofingBenchmarks/*`).

In [ ]:
# ---------------- Setup: clone/cập nhật repo, cài thư viện ----------------
import os, subprocess, sys
REPO_URL = 'https://github.com/trinhphamtrunghieu/nes2net-fake-voice-detection.git'
REPO_REF = 'main'   # ghim một commit/tag (vd '154b3b4') cho các lần train dài để code không đổi giữa các phiên
IN_COLAB, IN_KAGGLE = 'google.colab' in sys.modules, os.path.isdir('/kaggle/working')
REPO_DIR = ('/content/nes2net-fake-voice-detection' if IN_COLAB else
            '/tmp/nes2net-fake-voice-detection' if IN_KAGGLE else os.path.abspath('nes2net-fake-voice-detection'))
if 'nes2net' in sys.modules:
    print('!! nes2net đã được import trong runtime này: Runtime → Restart session rồi chạy lại để nạp code mới')
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', REPO_REF], check=True)
if REPO_REF == 'main':
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q', 'origin', 'main'], check=True)
print('code @', subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', os.path.join(REPO_DIR, 'requirements.txt')], check=True)
sys.path.insert(0, REPO_DIR)

In [2]:
# ---------------- Config ----------------
PRESET = 'paper_train'   # 'smoke' | 'default' | 'full_eval' | 'vi_zeroshot' | 'diag_data' | 'diag_train' | 'paper_train'
OVERRIDES = {}       # ghi đè bất kỳ khoá nào, vd {'B': {'lr': 2e-6}, 'run': {'L': False}}
USE_DRIVE = True     # Colab: lưu WORK lên Google Drive

from nes2net import env as nenv, presets, run
from IPython.display import display
ENV = nenv.setup(use_drive=USE_DRIVE)
cfg = presets.get(PRESET, ENV.platform, OVERRIDES)
print(ENV); presets.show(cfg)

Mounted at /content/drive
platform=colab device=cuda workers=12
  WORK=/content/drive/MyDrive/nes2net
  TMP =/content/nes2net_tmp
preset=paper_train platform=colab parts=['sanity', 'B']
  B: lr=2.5e-07 bs=12 epochs=100 algo=4 dev=asv19:validation time_budget_h=11.5 keep_best_k=3


"preset=paper_train platform=colab parts=['sanity', 'B']\n  B: lr=2.5e-07 bs=12 epochs=100 algo=4 dev=asv19:validation time_budget_h=11.5 keep_best_k=3\n"

## 0. Kiểm tra nhanh: back-end khớp bit-for-bit với code gốc, số tham số ≈ 511k

In [3]:
if cfg['run']['sanity']:
    display(run.sanity(ENV))

........                                                                 [100%]
8 passed in 28.74s

paper (Table III): Nes2Net 511k / 58.11 MMACs, Nes2Net-X 511k / 91.35 MMACs


,backend,params_k,MMACs_4s
0,nes2net,511.8,53.9
1,nes2net_x,512.0,84.2
2,dr_res2net,179.0,34.8


## A. Checkpoint chính thức → đánh giá
`load_official_checkpoint` chuyển key fairseq sang HF `Wav2Vec2Model` (không cần fairseq) và nạp **strict**.
Giao thức: cửa sổ 4 s đầu như code gốc; ITW chấm thêm cả câu. ASVspoof 2021: script chấm chính thức chỉ tính
trial `phase == eval` (con số của paper); Arena tính **toàn bộ** trial. Preset `full_eval` báo cả hai, kèm EER theo codec/vocoder.

In [4]:
if cfg['run']['A']:
    RA = run.eval_official(cfg, ENV)
    display(run.results_table(RA))
if cfg['run']['A_full']:
    RAF = run.eval_official_full(cfg, ENV)       # chạy lại được: shard đã chấm sẽ bỏ qua
    display(run.results_table(RAF))
    for k, v in RAF.items():
        if 'phase_eval_breakdown_eer' in v:
            print(k, {a: round(e, 2) for a, e in v['phase_eval_breakdown_eer'].items()})

## B. Train end-to-end
XLS-R 300M fine-tune + Nes2Net-X `[8,8]`, SE 1, dilation 2, mean pooling, Adam (wd 1e-4), weighted CE `[0.1, 0.9]`,
RawBoost algo 4, cửa sổ 4 s đầu; lưu checkpoint khi val loss (có/không aug) giảm.

**Resume:** trạng thái ghi vào `WORK/runB/last_state.pth` mỗi `save_every_steps` bước và cuối mỗi epoch; mỗi phiên dừng
an toàn sau `time_budget_h`. Bị ngắt hoặc hết giờ → chạy lại notebook với cùng preset. Trên Kaggle, thêm output của
lần chạy trước làm input của kernel: `runB/` được copy về tự động. Khi đủ epoch: `OVERRIDES = {'run': {'B': False, 'B_eval': True}}`.

In [ ]:
if cfg['run']['B']:
    HIST = run.train(cfg, ENV)
if cfg['run']['B_eval']:
    display(run.eval_trained(cfg, ENV))

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/421 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
wav2vec2.masked_spec_embed   | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.27GB            

... (8 outputs left out)

Generating train split:   0%|          | 0/25380 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/24844 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/71237 [00:00<?, ? examples/s]

17:31:41 INFO  nes2net.train | resumed from /content/drive/MyDrive/nes2net/runB/last_state.pth: epoch 7, step 2000
17:31:43 INFO  nes2net.train | start: epochs 7..100, 2115 steps/epoch (bs 12 x accum 1), train 25380 / dev 24844 utts, lr 2.5e-07, RawBoost algo 4, params 315.9M (315.9M trainable), device cuda (NVIDIA L4), amp True, workers 12, eval_bs 48, grad_ckpt False, attn sdpa
17:32:43 INFO  nes2net.train | ep 7 2050/2115 | loss 0.4572 (ep avg 0.4805) | lr 2.50e-07 | 9.9 utt/s | data 11% | gpu 3.5/8.1 GB | ETA ep 1.3 min
17:33:17 INFO  nes2net.train | ep 7 2100/2115 | loss 0.4431 (ep avg 0.4796) | lr 2.50e-07 | 18.0 utt/s | data 0% | gpu 3.5/8.1 GB | ETA ep 0.2 min
17:50:40 INFO  nes2net.train | == epoch 7/100 | train loss 0.4793 | val loss 1.5936 / aug 1.6967 | val EER 50.16% / aug 39.87% | best 0.6657 / 0.6588 | not saved | train 1.7 + val 17.2 min
17:51:40 INFO  nes2net.train | ep 8 50/2115 | loss 0.4497 (ep avg 0.4497) | lr 2.50e-07 | 12.4 utt/s | data 9% | gpu 3.5/8.1 GB | ETA 

### Log của phần B
`WORK/runB/train.log` (dạng chữ), `metrics.jsonl` (JSON từng sự kiện), `tb/` (TensorBoard). Cứ `log_every` bước một
dòng gồm loss, lr, utt/s, % thời gian chờ DataLoader, GPU memory, ETA; cuối epoch: val loss và val EER.

In [ ]:
SHOW_TB = False
if cfg['run']['B'] or os.path.exists(os.path.join(ENV.work, cfg['B']['out'], 'metrics.jsonl')):
    run.plot_train_logs(os.path.join(ENV.work, cfg['B']['out']), show_tb=SHOW_TB)

## C. Ablation back-end trên đặc trưng XLS-R đóng băng
Đặc trưng XLS-R (pre-trained, không fine-tune) của cửa sổ 4 s được cache fp16 một lần; mỗi back-end train vài phút.
EER tuyệt đối cao hơn paper, chỉ so sánh **tương đối** giữa các biến thể (mean ± std qua seed).
Nhóm A0–A5: baseline, tỉ lệ lồng, SE, pooling, dilation/kernel, bỏ kết nối tích luỹ (xem `docs/ablation_and_gaps.md`).

In [ ]:
if cfg['run']['C']:
    from nes2net.ablation import summarize
    DFC = run.ablation(cfg, ENV)
    display(summarize(DFC))
    run.plot_ablation(DFC, ENV)

### C2. Chọn layer SSL (research gap G1): từng layer so với tổng có trọng số học được

In [ ]:
if cfg['run']['L']:
    DFL = run.layer_study(cfg, ENV)
    display(DFL.groupby('layers')[['dev_eer', 'itw_eer']].agg(['mean', 'std']).round(2))

## DIAG. Kiểm tra dữ liệu train
Bản ASVspoof2019 LA dùng để train (`Bisher/ASVspoof_2019_LA`) có đúng như mong đợi không?
1. Bảng chéo nhãn × attack id: câu bona fide phải có attack `-`.
2. Thống kê audio theo split × lớp: định dạng lưu, sample rate, độ dài, mức âm lượng, khoảng lặng đầu/cuối.
3. EER của checkpoint chính thức trên một tập con của train / dev / test.

Nếu checkpoint chính thức chấm tốt test nhưng chấm kém train hoặc dev thì lỗi nằm ở bản dữ liệu, không phải code train.
Chạy cùng preset khác: `OVERRIDES = {'run': {'DIAG': True}}`.

In [ ]:
if cfg['run'].get('DIAG'):
    for t in run.diag_tables(run.diag_data(cfg, ENV)):
        display(t)

## DT. Kiểm tra vòng train
Các lần chạy ngắn bằng đúng `engine.train` (công thức paper, tập con train), rồi chấm EER trên train/dev ở ba chế độ: `eval`, `bn_batch` (BatchNorm dùng thống kê của batch), `train`.
- Checkpoint chính thức xấu đi sau 1 epoch → vòng train làm hỏng mô hình.
- `eval` xấu nhưng `bn_batch` tốt → lỗi ở running stats của BatchNorm.
- Bản AMP xấu, bản fp32 tốt → lỗi số học fp16.
- Cột `frontend` cho biết XLS-R pretrained có nạp đủ trọng số không.

In [ ]:
if cfg['run'].get('DT'):
    display(run.diag_train_table(run.diag_train(cfg, ENV)))

## VI. Zero-shot trên tiếng Việt
Checkpoint chính thức (train trên tiếng Anh, ASVspoof 2019 LA) chấm trên tập thử ghép đôi theo câu:
giọng thật từ **FLEURS `vi_vn`** và **cùng câu đó** do **MMS-TTS tiếng Việt** (VITS) đọc. Một tập đối chứng
tiếng Anh dựng y hệt (FLEURS `en_us` + MMS-TTS eng) giúp tách phần tụt do *ngôn ngữ* khỏi phần do *cách dựng tập*.
Mỗi tập chấm hai phiên bản: `raw`, và `norm` (cùng mức âm lượng, để mô hình không dựa vào độ to nhỏ).
Đây là phép thử nhanh (1 hệ TTS, giọng đọc) chứ không phải benchmark; SEA-Spoof / VSASV (cần xin quyền) sẽ thay thế.

In [ ]:
if cfg['run'].get('VI'):
    RVI = run.vi_zeroshot(cfg, ENV)
    display(run.vi_table(RVI))

## Kết quả trong `WORK`
`partA_results.json` · `partA_full_results.json` + `A_full/` · `runB/` · `partB_results.json` ·
`partC_ablation.csv/.png` · `partC_layers.csv` · `partVI_results.json`. Dòng lệnh tương đương: `python -m nes2net.run --preset <tên>`.